In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from helpers.retrieval import search_chunks_local
from helpers.mlflow_retriever import load_retriever_artifacts

In [0]:
spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA med")

In [0]:
RUN_ID = "4ea1952dd36348f997c9791e6c5fadc4"
vectorizer, knn, chunk_ids, metadata = load_retriever_artifacts(RUN_ID)

In [0]:
# load chunk data in same order used during training
chunks_df = spark.table("workspace.med.doc_chunks")
chunks_pdf = chunks_df.select(
    "chunk_id",
    "doc_id",
    "chunk_text",
    "source",
    "category",
    "title"
).toPandas()

In [0]:
# test queries
questions = [
    "what is ibuprofen used for",
    "can i take ibuprofen with food",
    "what are side effects of nsaids",
]

for q in questions:
    print("\nQUESTION:", q)
    results = search_chunks_local(
        q,
        vectorizer=vectorizer,
        knn=knn,
        chunks_pdf=chunks_pdf,
        top_k=5,
    )

    for r in results:
        print(r["rank"], r["title"], r["cosine_distance"])
        print(r["chunk_text_preview"])
        print("-" * 80)